# Comunicação com stakeholders (item 5) — cálculos, versão Python

Esta é a bancada de cálculo por trás do relatório de negócio (item 5) — as mesmas contas que sustentam [`docs/05_comunicacao_stakeholders.html`](../docs/05_comunicacao_stakeholders.html), com o código à mostra, em cima do **modelo de referência do projeto (XGBoost, `analysis-python/02-1_modelagem.ipynb`)**.

Existe uma versão equivalente em R, [`analysis-r/05_comunicacao_stakeholders_calculos.qmd`](../analysis-r/05_comunicacao_stakeholders_calculos.qmd) — mas ela roda em cima da Random Forest daquela implementação alternativa, então os números de lá (erro típico, comparação de modelos) **não são os mesmos** desta página. Esta aqui é a que efetivamente sustenta o relatório final.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATA = "../data"
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Reconstrução do dataset e do split

Mesma preparação e o mesmo `random_state=42` de [`02-1_modelagem.ipynb`](02-1_modelagem.ipynb) — então `X_test`/`y_test` aqui são exatamente o mesmo conjunto de teste usado pra treinar e avaliar o modelo salvo, o que permite reaproveitá-lo sem retreinar.

In [2]:
casas = pd.read_csv(f"{DATA}/kc_house_data.csv")
casas["date"] = pd.to_datetime(casas["date"], format="%Y%m%dT%H%M%S")
casas = casas[casas["bedrooms"] < 30].copy()

demograf = pd.read_csv(f"{DATA}/zipcode_demographics.csv")

casas["log_price"] = np.log(casas["price"])
casas["tem_porao"] = (casas["sqft_basement"] > 0).astype(int)
casas["idade_casa"] = casas["date"].dt.year - casas["yr_built"]
casas["reformado"] = (casas["yr_renovated"] > 0).astype(int)
casas = casas.merge(demograf, on="zipcode", how="left")

vars_modelo = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "grade", "tem_porao", "idade_casa", "reformado",
    "lat", "long", "sqft_living15", "sqft_lot15",
    "medn_hshld_incm_amt", "medn_incm_per_prsn_amt", "hous_val_amt", "per_bchlr", "per_prfsnl"]

X = casas[vars_modelo]
y = casas["log_price"]
bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=bins)

modelo_xgb = joblib.load("models/modelo_xgb.joblib")
pred_xgb_teste = modelo_xgb.predict(X_test)

mediana_preco = casas["price"].median()
mae_xgb_dolar = mean_absolute_error(np.exp(y_test), np.exp(pred_xgb_teste))
mae_xgb_pct = mae_xgb_dolar / mediana_preco

print("imóveis usados para aprender:", casas.shape[0])
print("mediana do preço:", mediana_preco)
print("MAE (US$):", mae_xgb_dolar)
print("MAE (%):", mae_xgb_pct * 100)

imóveis usados para aprender: 21612
mediana do preço: 450000.0
MAE (US$): 63706.76711774231
MAE (%): 14.157059359498291


## 2. Modelo de referência para comparação (regressão linear)

Mesmo espírito do script em R: comparo o XGBoost contra a abordagem mais simples que testei ([`02-1_modelagem.ipynb`](02-1_modelagem.ipynb)), pra ter um número de "quanto melhor" que não seja só o RMSE em escala log.

In [3]:
lm = LinearRegression().fit(X_train, y_train)
pred_lm_teste = lm.predict(X_test)
mae_lm_dolar = mean_absolute_error(np.exp(y_test), np.exp(pred_lm_teste))

print("MAE regressão linear (US$):", mae_lm_dolar)
print("Diferença (US$):", mae_lm_dolar - mae_xgb_dolar)

MAE regressão linear (US$): 90351.60768499743
Diferença (US$): 26644.840567255123


## 3. Efeitos marginais (a partir dos coeficientes da regressão linear)

Mesma lógica do script em R: uso os coeficientes do modelo mais simples e interpretável — não do XGBoost, cujos efeitos não são lineares — para traduzir "cada unidade a mais de X está associada a Y% a mais no preço, em média, mantendo o resto constante". Como o alvo é `log(price)`, o efeito percentual de um coeficiente `b` é `(exp(b) − 1) × 100`.

In [4]:
coefs = pd.Series(lm.coef_, index=vars_modelo)

efeito_grade_pct = (np.exp(coefs["grade"]) - 1) * 100
efeito_sqft100_pct = (np.exp(coefs["sqft_living"] * 100) - 1) * 100

print(f"Efeito de +1 ponto em grade: {efeito_grade_pct:.1f}%")
print(f"Efeito de +100 pés² em sqft_living: {efeito_sqft100_pct:.1f}%")

Efeito de +1 ponto em grade: 11.2%
Efeito de +100 pés² em sqft_living: 1.7%


## 4. O que mais pesa no preço (importância de variáveis do XGBoost)

Uso a importância nativa do modelo salvo (`gain`: o quanto cada variável contribui, em média, pra reduzir o erro quando é usada num split) — a mesma tabela já calculada em [`02-1_modelagem.ipynb`](02-1_modelagem.ipynb#7.-Importância-das-variáveis), só reaproveitada aqui.

In [5]:
rotulos = {
    "grade": "Padrão de acabamento/construção",
    "per_prfsnl": "% de moradores com ocupação de nível superior no bairro",
    "hous_val_amt": "Valor médio dos imóveis no bairro",
    "per_bchlr": "% de moradores com diploma universitário no bairro",
    "sqft_living": "Tamanho da casa",
    "lat": "Localização (norte–sul)",
}

importancia = pd.Series(modelo_xgb.feature_importances_, index=vars_modelo).sort_values(ascending=False)
top6 = importancia.head(6).rename(index=rotulos) * 100
top6.round(1)

Padrão de acabamento/construção                           30.1000
% de moradores com ocupação de nível superior no bairro   19.5000
Valor médio dos imóveis no bairro                         12.0000
% de moradores com diploma universitário no bairro        11.9000
Tamanho da casa                                            6.6000
Localização (norte–sul)                                    5.0000
dtype: float32

Três dos seis fatores mais fortes não são características do imóvel em si, são do bairro (% ocupação de nível superior, valor médio dos imóveis, % diploma universitário) — o modelo está aprendendo tanto "essa casa é boa" quanto "esse bairro é valorizado".

## 5. Onde o modelo é menos confiável (item 2.2)

Não retreino a bateria completa de generalização aqui — ela já está em [`02-2_generalizacao.ipynb`](02-2_generalizacao.ipynb), com o código completo (bootstrap, split temporal, CV agrupada por CEP). Só reaproveito os dois números que viram a recomendação de negócio:

- Split temporal (treina no passado, testa no futuro): RMSE piora **+13,0%** em relação ao split aleatório.
- CV agrupada por CEP (bairro nunca visto no treino): RMSE piora **+20,5%** — a maior fraqueza do modelo não é "ficar desatualizado com o tempo", é avaliar um bairro novo ou raramente vendido.

Uma terceira ressalva, essa sim calculada diretamente aqui em cima do XGBoost (não herdada da versão R): o modelo **subestima sistematicamente os imóveis mais caros**. Diagnóstico de resíduos completo em [`02-1_modelagem.ipynb`](02-1_modelagem.ipynb#8.-Diagn%C3%B3stico-dos-res%C3%ADduos-(modelo-final)) — viés médio de **+US\$ 78.004** no top 10% mais caro da base de teste (real acima do previsto), contra viés próximo de zero (-US\$ 3.222) no restante. Limitação conhecida de modelos baseados em árvore: a previsão é uma média/combinação de folhas vistas no treino, com dificuldade de extrapolar para a cauda superior da distribuição.

## 6. Resumo dos números usados no relatório final

In [6]:
resumo = pd.DataFrame({
    "métrica": [
        "vendas usadas para aprender", "erro típico (US$)", "erro típico (%)",
        "erro típico, regressão linear (US$)", "diferença vs. regressão linear (US$)",
        "efeito de +1 grade", "efeito de +100 pés² em sqft_living",
        "degradação em split temporal", "degradação em CEP novo",
    ],
    "valor": [
        f"{casas.shape[0]:,}", f"US$ {mae_xgb_dolar:,.0f}", f"{mae_xgb_pct * 100:.1f}%",
        f"US$ {mae_lm_dolar:,.0f}", f"US$ {mae_lm_dolar - mae_xgb_dolar:,.0f}",
        f"{efeito_grade_pct:.1f}%", f"{efeito_sqft100_pct:.1f}%",
        "+13,0% (item 2.2)", "+20,5% (item 2.2)",
    ],
})
resumo

,métrica,valor
0,vendas usadas para aprender,"21,612"
1,erro típico (US$),"US$ 63,707"
2,erro típico (%),14.2%
3,"erro típico, regressão linear (US$)","US$ 90,352"
4,diferença vs. regressão linear (US$),"US$ 26,645"
5,efeito de +1 grade,11.2%
6,efeito de +100 pés² em sqft_living,1.7%
7,degradação em split temporal,"+13,0% (item 2.2)"
8,degradação em CEP novo,"+20,5% (item 2.2)"
